# Deep Learning: Tensor Mathematics

Welcome to **Section 9: Deep Learning & Neural Networks**.

Up to this point, we have relied on `pandas` DataFrames and `numpy` arrays. These structures are excellent for traditional Machine Learning on a CPU. However, Deep Learning requires calculating millions (or billions) of mathematical gradients simultaneously. A standard CPU cannot handle this. We must move our calculations to a GPU (Graphics Processing Unit).

To do this, we must abandon standard arrays and adopt a new mathematical container designed specifically for extreme parallel computation: the **Tensor**. In this lesson, we will use **PyTorch**, the undisputed enterprise standard for Deep Learning engineering.

A Tensor is simply a generalized mathematical container for numerical data. The fundamental difference between a Tensor and a `numpy` array is that Tensors are engineered to live in GPU VRAM (Video RAM) and natively track gradients for calculus operations.

Let's set up our Deep Learning environment.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print(f"✅ PyTorch Version {torch.__version__} Ready.")

✅ PyTorch Version 2.5.1+cu121 Ready.


# 1. The Geometry of Dimensions (Rank)

Tensors are categorized by their **Rank** (number of dimensions or axes).

### 0D Tensor: Scalar

A single number. It has magnitude, but no direction.

* *Math*: $x \in \mathbb{R}$
* *Example*: The learning rate, or the final calculated Loss of a neural network.

### 1D Tensor: Vector

An array of numbers. It has magnitude and direction along one axis.

* *Math*: $x \in \mathbb{R}^n$
* *Example*: A single row of a dataset, or a time series of temperatures over a week.

### 2D Tensor: Matrix

A grid of numbers with rows and columns.

* *Math*: $X \in \mathbb{R}^{m \times n}$
* *Example*: A standard tabular dataset (like a CSV file) or a grayscale image (Height $\times$ Width).

### 3D Tensor and Beyond

An array of matrices.

* *Math*: $X \in \mathbb{R}^{c \times h \times w}$
* *Example 3D*: A color image (3 color channels: Red, Green, Blue $\times$ Height $\times$ Width).
* *Example 4D*: A batch of color images processed simultaneously (Batch_Size $\times$ Channels $\times$ Height $\times$ Width).

In [2]:
# 1. Creating Tensors of different Ranks
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
tensor_3d = torch.rand(size=(3, 224, 224)) # E.g., A random 224x224 RGB image

print("--- Tensor Geometry ---")
print(f"Scalar Shape:   {scalar.shape} | Dimensions: {scalar.ndim}")
print(f"Vector Shape:   {vector.shape}    | Dimensions: {vector.ndim}")
print(f"Matrix Shape:   {matrix.shape} | Dimensions: {matrix.ndim}")
print(f"3D Tensor Shape:{tensor_3d.shape} | Dimensions: {tensor_3d.ndim}")

--- Tensor Geometry ---
Scalar Shape:   torch.Size([]) | Dimensions: 0
Vector Shape:   torch.Size([3])    | Dimensions: 1
Matrix Shape:   torch.Size([3, 2]) | Dimensions: 2
3D Tensor Shape:torch.Size([3, 224, 224]) | Dimensions: 3


# 2. Tensor Operations & Matrix Multiplication

Neural Networks are, at their core, just billions of multiplications and additions.
When we multiply a matrix of data ($X$) by a matrix of Neural Network weights ($W$), we use the **Dot Product** (Matrix Multiplication).

For matrix multiplication $C = A \cdot B$ to be mathematically possible, the inner dimensions must match. If $A$ is $(m \times n)$, $B$ must be $(n \times p)$. The resulting matrix $C$ will be $(m \times p)$.


$$C_{i,j} = \sum_{k=1}^{n} A_{i,k} B_{k,j}$$

In [3]:
# Create a Dataset (3 samples, 4 features)
X = torch.tensor([[1, 2, 3, 4],
                  [5, 6, 7, 8],
                  [9, 10, 11, 12]], dtype=torch.float32) # Shape: (3, 4)

# Create Neural Network Weights (4 features, 2 output neurons)
W = torch.tensor([[0.1, 0.2],
                  [0.3, 0.4],
                  [0.5, 0.6],
                  [0.7, 0.8]], dtype=torch.float32) # Shape: (4, 2)

# Matrix Multiplication using torch.matmul (or the @ operator)
# (3x4) @ (4x2) = (3x2)
output = X @ W 

print("\n--- Matrix Multiplication ---")
print(f"Data X Shape: {X.shape}")
print(f"Weights W Shape: {W.shape}")
print(f"Output Shape: {output.shape}")
print("Resulting Tensor:\n", output)


--- Matrix Multiplication ---
Data X Shape: torch.Size([3, 4])
Weights W Shape: torch.Size([4, 2])
Output Shape: torch.Size([3, 2])
Resulting Tensor:
 tensor([[ 5.0000,  6.0000],
        [11.4000, 14.0000],
        [17.8000, 22.0000]])


# 3. The Magic of Broadcasting

What happens if you try to add a 1D Vector to a 2D Matrix? In strict linear algebra, this is illegal.

However, Deep Learning frameworks use **Broadcasting**. Broadcasting dynamically and virtually "stretches" the smaller tensor to match the shape of the larger tensor, *without actually allocating new memory in RAM*.

In [4]:
# A 2D Matrix (3 rows, 3 columns)
A = torch.tensor([[1, 1, 1],
                  [2, 2, 2],
                  [3, 3, 3]])

# A 1D Vector (1 row, 3 columns)
B = torch.tensor([10, 20, 30])

# Broadcasting in action:
# PyTorch realizes B doesn't have enough rows. It virtually copies B three times:
# [[10, 20, 30],
#  [10, 20, 30],
#  [10, 20, 30]]
# Then it adds them element-wise!
C = A + B

print("\n--- Broadcasting ---")
print("Matrix A:\n", A)
print("Vector B:\n", B)
print("Result C (A + B):\n", C)


--- Broadcasting ---
Matrix A:
 tensor([[1, 1, 1],
        [2, 2, 2],
        [3, 3, 3]])
Vector B:
 tensor([10, 20, 30])
Result C (A + B):
 tensor([[11, 21, 31],
        [12, 22, 32],
        [13, 23, 33]])


*(Insight: Broadcasting saves massive amounts of memory. If you want to add a bias term to 10,000 images, you don't need to create a bias tensor of 10,000 images; you just create one small bias vector and let PyTorch broadcast it across the entire batch!)*

# 4. Hardware Acceleration: The CUDA Device

The true superpower of Tensors is where they physically exist inside your computer.

* **CPUs (Central Processing Units)**: Have a few very fast cores (e.g., 8 to 16). They are optimized for sequential logic (if/else statements). Calculating a million matrix multiplications on a CPU takes minutes.
* **GPUs (Graphics Processing Units)**: Have thousands of relatively slow cores (e.g., 10,000+ CUDA cores). They are heavily optimized for doing the exact same simple math problem simultaneously. Calculating a million matrix multiplications on a GPU takes milliseconds.

To unleash this power, we must physically move the tensor out of the computer's standard RAM, travel across the motherboard's PCIe bus, and drop it into the GPU's VRAM.

In [5]:
print("\n--- GPU Acceleration ---")

# 1. Check if a GPU is physically present and accessible by PyTorch
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🚀 NVIDIA GPU Detected: {torch.cuda.get_device_name(0)}")
# Check for Apple Silicon (M1/M2/M3 MacBooks)
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🚀 Apple Silicon GPU Detected.")
else:
    device = torch.device('cpu')
    print("🐢 No GPU Detected. Falling back to CPU.")

# 2. Create a massive Tensor
massive_tensor = torch.rand(10000, 10000)

# 3. Physically move the Tensor to the GPU VRAM
massive_tensor_gpu = massive_tensor.to(device)

print(f"Tensor Location: {massive_tensor_gpu.device}")


--- GPU Acceleration ---
🚀 NVIDIA GPU Detected: NVIDIA GeForce GTX 1650 Ti
Tensor Location: cuda:0


*(Enterprise Rule: If Tensor A is on the CPU, and Tensor B is on the GPU, you cannot multiply them. PyTorch will crash. You must explicitly ensure all interacting data and neural network weights live on the exact same hardware device).*

## Real-World Use Case or Analogy:

Think of the difference between CPUs and GPUs (and why Tensors are required) like **Transporting Cargo**:

* **CPU (The Ferrari)**: You need to move 10,000 boxes from New York to Boston. The CPU is a Ferrari. It is incredibly fast. It puts one box in the passenger seat, drives to Boston at 200 MPH, drops it off, and drives back. It takes days to move all 10,000 boxes because it processes them sequentially.
* **GPU (The Cargo Ship)**: A GPU is a massive cargo ship. It is very slow (20 MPH). But it can hold all 10,000 boxes simultaneously. It takes one single trip to move the entire dataset.
* **The Tensor (The Shipping Container)**: You cannot just throw loose boxes onto a cargo ship. You must pack them into standardized, mathematically rigid Shipping Containers. Tensors are the shipping containers that allow the GPU to load, process, and offload massive amounts of data in perfectly parallelized chunks.

---